![Snowflake](https://www.snowflake.com/wp-content/themes/flavor/assets/img/logo-snowflake-sans-word.svg)
# CredibanCo — Hands-On Lab
**RFP 10010806** · Plataforma de Datos · Snowflake 2026

---

# Track 6 — Marketplace, Autoservicio y BI
**Rol:** CRB_NEGOCIO | **Tiempo:** 15 min | **Criterio:** Descubrir, solicitar, aprovisionar y consumir datos con autonomía

In [ ]:
USE ROLE CRB_NEGOCIO;
USE DATABASE CREDIBANCO_HOL;
USE WAREHOUSE CREDIBANCO_HOL_WH;

## Bloque 1 — Evidencia: Autoservicio y BI funcionando

Verificamos que existen **sandboxes auto-aprovisionados**. Cada usuario de negocio puede crear su espacio de exploración sin depender de IT — autoservicio real.

In [ ]:
-- Sandbox auto-aprovisionado (el usuario puede crear su espacio)
SHOW SCHEMAS LIKE 'SANDBOX%' IN DATABASE CREDIBANCO_HOL;

Consultamos datos de turismo disponibles como **producto de datos** para análisis de negocio. El dominio de Monetización expone datasets listos para consumo por equipos de negocio.

In [ ]:
-- Datos de turismo disponibles para análisis de negocio
SELECT REGION, SUM(MONTO_TOTAL) AS MONTO, SUM(NUM_TRANSACCIONES) AS TXS
FROM CREDIBANCO_HOL.TURISMO.AGREGADOS_TURISMO
GROUP BY 1 ORDER BY MONTO DESC;

El catálogo de productos de datos muestra qué datasets están disponibles y cuántos registros tiene cada uno. Los usuarios de negocio descubren datos sin necesidad de solicitar acceso a IT.

In [ ]:
-- Inventario de productos de datos disponibles
SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE, ROW_COUNT
FROM CREDIBANCO_HOL.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('MONETIZACION','TURISMO') AND ROW_COUNT > 0
ORDER BY TABLE_SCHEMA, TABLE_NAME;

## Bloque 2 — Ejecutar: Query de negocio + Cortex Analyst

In [ ]:
-- Análisis de negocio: top regiones × categoría
SELECT REGION, MCC, MONTO_TOTAL,
       RANK() OVER (PARTITION BY REGION ORDER BY MONTO_TOTAL DESC) AS rank_mcc
FROM CREDIBANCO_HOL.TURISMO.AGREGADOS_TURISMO
QUALIFY rank_mcc <= 3
ORDER BY REGION, rank_mcc;

### Cortex Analyst — Preguntas en Lenguaje Natural
**Cortex Analyst** permite a usuarios de negocio hacer preguntas en español y obtener respuestas con datos reales — sin escribir SQL. Primero creamos el modelo semántico:

In [ ]:
-- Crear Semantic View para Cortex Analyst
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE SEMANTIC VIEW CREDIBANCO_HOL.PAGOS.SV_AUTORIZACIONES
  TABLES (
    aut AS CREDIBANCO_HOL.PAGOS.AUTORIZACIONES
      PRIMARY KEY (AUTORIZACION_ID)
      COLUMNS (
        AUTORIZACION_ID COMMENT 'ID único de la autorización',
        COMERCIO_ID COMMENT 'ID del comercio',
        CIUDAD COMMENT 'Ciudad donde se realizó la transacción',
        MCC COMMENT 'Código de categoría del comercio',
        MONTO COMMENT 'Monto de la transacción en pesos colombianos',
        CODIGO_RESPUESTA COMMENT '00=aprobada, otro=rechazada',
        CANAL COMMENT 'Canal: POS, ECOMMERCE, ATM',
        FECHA_HORA COMMENT 'Fecha y hora de la transacción'
      )
  )
  RELATIONSHIPS ()
  FACTS (
    aut.MONTO,
    aut.AUTORIZACION_ID
  )
  DIMENSIONS (
    aut.CIUDAD,
    aut.MCC,
    aut.CODIGO_RESPUESTA,
    aut.CANAL
  );

In [ ]:
-- Probar Cortex Analyst: pregunta en lenguaje natural
SELECT SNOWFLAKE.CORTEX.ANALYST(
  '¿Cuáles son las 5 ciudades con más transacciones aprobadas y cuál es el monto promedio?',
  OBJECT_CONSTRUCT('semantic_view', 'CREDIBANCO_HOL.PAGOS.SV_AUTORIZACIONES')
);

El usuario de negocio preguntó en español y Cortex Analyst generó el SQL, lo ejecutó y retornó los resultados — **sin que nadie escribiera una query**.

## Bloque 2B — Publicar un Producto de Datos en el Internal Marketplace
Creamos un **data product** y lo publicamos en el Internal Marketplace para que otras áreas de CredibanCo puedan consumirlo sin mover datos.

In [ ]:
-- Crear un share con las tablas de pagos y comercios
CREATE SHARE IF NOT EXISTS CREDIBANCO_PAGOS_SHARE
  COMMENT = 'Producto de datos: Autorizaciones de pagos CredibanCo';

-- Otorgar acceso en orden: DB → Schema → Tablas
GRANT USAGE ON DATABASE CREDIBANCO_HOL TO SHARE CREDIBANCO_PAGOS_SHARE;
GRANT USAGE ON SCHEMA CREDIBANCO_HOL.PAGOS TO SHARE CREDIBANCO_PAGOS_SHARE;
GRANT SELECT ON TABLE CREDIBANCO_HOL.PAGOS.AUTORIZACIONES TO SHARE CREDIBANCO_PAGOS_SHARE;
GRANT SELECT ON TABLE CREDIBANCO_HOL.PAGOS.LIQUIDACIONES TO SHARE CREDIBANCO_PAGOS_SHARE;
GRANT USAGE ON SCHEMA CREDIBANCO_HOL.COMERCIOS TO SHARE CREDIBANCO_PAGOS_SHARE;
GRANT SELECT ON TABLE CREDIBANCO_HOL.COMERCIOS.COMERCIOS TO SHARE CREDIBANCO_PAGOS_SHARE;

Verificamos el contenido del share — 3 tablas compartidas **sin copiar datos**. Zero-copy sharing es una ventaja única de Snowflake.

In [ ]:
-- Verificar el contenido del share
DESCRIBE SHARE CREDIBANCO_PAGOS_SHARE;

Ahora publica el listing desde la UI:

1. Ve a **Data Sharing > Internal Sharing > + Create Listing**
2. Selecciona el share `CREDIBANCO_PAGOS_SHARE`
3. Llena título: **CredibanCo — Autorizaciones de Pagos y Comercios**
4. Selecciona los custom attributes (Confidentiality, Source Systems)
5. Publica

El listing aparecerá en el **Internal Marketplace** para toda la organización.

## Bloque 2C — Streamlit: App de BI en Snowflake
Streamlit permite crear **aplicaciones interactivas** directamente dentro de Snowflake — sin servidores, sin deploy externo. El código, los datos y la app viven en la misma plataforma.

Copia este prompt en **Cortex Code**:

> **Crea una app Streamlit en Snowflake que muestre un dashboard de autorizaciones de CredibanCo con: (1) filtro por ciudad en sidebar, (2) KPI cards de total transacciones, monto total y tasa de aprobación, (3) gráfico de barras por MCC, (4) tabla con las últimas 100 transacciones. Usa la tabla CREDIBANCO_HOL.PAGOS.AUTORIZACIONES. Despliégala en Snowflake.**

Una vez creada, la app aparece en **Projects > Streamlit** en Snowsight. Cualquier usuario con el rol adecuado puede accederla sin instalar nada.

**Ventaja sobre Databricks:** Streamlit corre nativo en Snowflake — no necesita cluster, no necesita notebook, no necesita deploy manual. Los datos nunca salen de la plataforma.

## Bloque 3 — CoCo
Copia este prompt en Cortex Code:

> **Genera un dashboard React local autocontenido (HTML+CDN, sin npm) con los datos de CREDIBANCO_HOL.TURISMO.AGREGADOS_TURISMO. Incluye KPIs, bar chart por región, tabla con filtros. Estilo Snowflake (#29B5E8). Primero ejecuta la query para obtener los datos reales.**

In [ ]:
-- Verificación final
SELECT 'T6_COMPLETO' AS status;